**Carbon-14 radioactive decay via Euler's method.**

**Deriving the decay equation from the definition of half-life.**
Radioactive decay is proportional to the current number of nuclei,
so we write:

$$\frac{dn}{dt} = -\lambda\, n$$

where $\lambda$ is a positive constant (the decay constant) to be determined.
The analytic solution is:

$$n(t) = n_0\,e^{-\lambda t}$$

The **half-life** $t_{1/2}$ is defined as the time at which exactly half
the nuclei have decayed:

$$n(t_{1/2}) = \frac{n_0}{2}$$

Substituting the analytic solution:

$$\frac{n_0}{2} = n_0\,e^{-\lambda\,t_{1/2}}$$

Dividing both sides by $n_0$:

$$\frac{1}{2} = e^{-\lambda\,t_{1/2}}$$

Taking the natural logarithm of both sides:

$$\ln\!\left(\frac{1}{2}\right) = -\lambda\,t_{1/2}$$

$$-\ln 2 = -\lambda\,t_{1/2}$$

Solving for $\lambda$:

$$\lambda = \frac{\ln 2}{t_{1/2}}$$

Substituting back into $dn/dt = -\lambda\,n$ gives the decay equation
expressed directly in terms of the measurable half-life:

$$\frac{dn}{dt} = -\frac{\ln 2}{t_{1/2}}\,n$$

**Euler's method** approximates this ODE numerically by stepping forward in time:

$$n_{i+1} = n_i - \frac{\ln 2}{t_{1/2}}\,n_i\,\Delta t
\qquad
t_{i+1} = t_i + \Delta t$$

---
**Simulation parameters.**
The simulation runs from $t = 0$ to $t_f = 40{,}000$ years using $t_s = 100$
evenly spaced time steps, giving a step size of $\Delta t = 400$ years.
These three values fully determine the Euler integration grid.

In [ ]:
"""carbon14_decay.ipynb"""

# Cell 01 - Simulation parameters

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf = 40_000  # final time (years)
ts = 100  # number of time steps
dt = tf / ts  # time step size (400 years per step)

print(f"{tf=:,}  {ts=:,}  {dt=:.2f}")

**Initializing the time and concentration arrays.**
Both arrays are pre-allocated with `np.zeros(ts)` and filled incrementally
by the Euler loop in Cell 04.
The initial conditions are set in Cell 03: $n[0] = 100\%$ and $t[0] = 0$.

In [ ]:
# Cell 02 - Allocate arrays for time and nuclei concentration

t = np.zeros(ts)  # time stamps (years)
n = np.zeros(ts)  # nuclei concentration (% of initial)

pd.DataFrame({"t": t[:5], "n": n[:5]})

**Initial conditions.**
The half-life of Carbon-14 is $t_{1/2} = 5{,}730$ years.
The simulation starts with 100% concentration at $t = 0$.

In [ ]:
# Cell 03 - Initial conditions: 100% concentration, half-life of C-14

n[0] = 100  # initial concentration (100%)
tau = 5730  # half-life of C-14 (years)

pd.DataFrame({"t": t[:5], "n": n[:5]})

---
**Euler's method loop.**
At each step $i$, the next concentration is updated using the discrete form
of the decay equation:

$$n[i+1] = n[i] - \frac{\ln 2}{\tau}\,n[i]\,\Delta t$$

After one half-life ($t = 5{,}730$ years) the concentration should reach
approximately 50%, which serves as a sanity check on the result.

In [ ]:
# Cell 04 - Euler forward integration of the decay equation
# n[i+1] = n[i] - (ln2 / tau) * n[i] * dt
# The factor ln2/tau is the decay constant lambda

decay_const = np.log(2) / tau  # lambda = ln(2) / half-life

for i in range(ts - 1):
    t[i + 1] = t[i] + dt
    n[i + 1] = n[i] - decay_const * n[i] * dt

pd.DataFrame({"t": t[:5], "n": n[:5]})

**Plotting the decay curve.**
The analytic solution $n(t) = 100 \cdot 2^{-t/\tau}$ is overlaid as a dashed
line so the accuracy of Euler's method can be assessed.
With only 100 time steps over 40,000 years ($\Delta t = 400$ years)
Euler's method introduces a visible but small cumulative error.

In [ ]:
# Cell 05 - Plot Euler approximation vs analytic solution

t_exact = np.linspace(0, tf, 500)
n_exact = 100 * 2 ** (-t_exact / tau)

plt.figure("carbon14_decay", figsize=(8, 5))
plt.plot(t, n, label="Euler's method")
plt.plot(t_exact, n_exact, "--", label=r"Analytic $n_0\,2^{-t/\tau}$")
plt.title("Carbon-14 Decay")
plt.xlabel("Time (years)")
plt.ylabel("% Concentration")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**How far back can Carbon-14 dating reach?**
As of 2026, the most sensitive accelerator mass spectrometry (AMS) detectors
can measure Carbon-14 concentrations down to approximately **0.1%** of the
original amount.
Below this threshold the signal is indistinguishable from background noise.

Using the analytic solution $n(t) = n_0\,2^{-t/t_{1/2}}$, we can solve
for the maximum age $t_{\max}$ at which the remaining concentration
equals the detection limit $n_{\min}$:

$$n_{\min} = n_0\,2^{-t_{\max}/t_{1/2}}$$

$$\frac{n_{\min}}{n_0} = 2^{-t_{\max}/t_{1/2}}$$

Taking $\log_2$ of both sides:

$$\log_2\!\left(\frac{n_{\min}}{n_0}\right) = -\frac{t_{\max}}{t_{1/2}}$$

$$t_{\max} = -t_{1/2}\,\log_2\!\left(\frac{n_{\min}}{n_0}\right)
= \frac{t_{1/2}\,\ln(n_0/n_{\min})}{\ln 2}$$

With $n_{\min}/n_0 = 0.001$ (0.1%) this equals
$t_{1/2}\,\log_2(1000) \approx 9.97\,t_{1/2}$,
meaning we can look back just under **10 half-lives**.

In [ ]:
# Cell 06 - Maximum reliable dating age given a 0.1% detection limit

n_min_pct = 0.1  # detection limit (% of original)
t_max = -tau * np.log(n_min_pct / 100) / np.log(2)

print(f"Detection limit          : {n_min_pct}% of original C-14")
print(f"Half-lives elapsed       : {t_max / tau:.2f}")
print(f"Maximum dating age       : {t_max:,.0f} years")
print(f"                         : ~{t_max / 1000:.1f} thousand years ago")

# Show on the decay curve
t_plot = np.linspace(0, 70_000, 1000)
n_plot = 100 * 2 ** (-t_plot / tau)

plt.figure("carbon14_detection_limit", figsize=(9, 5))
plt.plot(t_plot, n_plot, label=r"$n(t) = 100\cdot 2^{-t/\tau}$")
plt.axhline(
    n_min_pct, color="red", linestyle="--", label=f"Detection limit ({n_min_pct}%)"
)
plt.axvline(t_max, color="orange", linestyle="--", label=f"Max age ({t_max:,.0f} yr)")
plt.scatter([t_max], [n_min_pct], color="red", zorder=5)
plt.title("Carbon-14 Dating Range (AMS detection limit 0.1%)")
plt.xlabel("Time (years)")
plt.ylabel("% C-14 remaining")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()